# 03 — Resultados de la etapa 2 (fine-tuning, VAL)

Análisis del ajuste QLoRA de la etapa 2 (CLAUDE.md §3; `docs/metodologia.md` §2.3 y §4). **Solo lee `runs/`**: no
entrena ni infiere. Reutiliza `scripts/summarize_train.py` (resumen de runs y regla de la grilla) y `vlmfid.eval`
(extractor y métricas), así que los números coinciden con los que imprimen los scripts.

**Todo es VAL** salvo §9, que solo *lee* la tabla que deja `scripts/final_test.py` cuando ya se corrió (test se usa
una sola vez; nada de este notebook debe influir en una decisión después de ver §9).

| § | Pregunta | Runs |
|---|---|---|
| 1 | Humo: ¿entrena lo que debe?, memoria pico (P8), ¿baja la pérdida? | `train_*_m6` |
| 2 | Grilla LR × r × conector: ganador por modelo y sensibilidad | `train_*_m150` |
| 3 | ¿Es robusto el ganador de la grilla? (bootstrap pareado, VLM) | `val_predictions/` de la grilla |
| 4 | Ajuste completo: curvas de entrenamiento y de val, paso elegido | `train_*_e1` |
| 5 | Val completo: zero-shot (`p1_attributes`) frente a ajustado, por atributo | `finetuned_*_val` y `zeroshot_*_p1_attributes_val` |
| 6 | Omitir frente a equivocarse y confusiones, antes y después del ajuste | ídem |
| 7 | Control bf16 después del ajuste (McNemar) | `finetuned_*_bf16_*_val` |
| 8 | VL-JEPA: Y-Decoder de la etapa 2 y pérdida de información, etapa 1 frente a etapa 2 | `ydec_*_ft_*`, `infoloss_*` |
| 9 | Tabla resumen (val) y, si existe, la tabla final de test | |

**Diferencias declaradas entre modelos** (`docs/metodologia.md` §6.1): VL-JEPA se selecciona por la InfoNCE en val
(mínima, P7), no por `attr_mean`, porque durante el ajuste no hay un Y-Decoder válido; la grilla elige un ganador
**por modelo** (G1). Las secciones se saltan solas si sus runs aún no existen: el notebook se puede volver a correr
a medida que avanzan las colas.

**Requisitos (cluster):** `scripts/score.py` corrido sobre `runs/finetuned_*_val` (métricas de texto). Kernel `vlmfid`.

In [ ]:
import json
import math
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from vlmfid.data.manifest import load_manifest
from vlmfid.eval import ATTRIBUTES, METRICS, TEXT_METRICS, extract, majority_baseline, score, truth_from_record
from vlmfid.eval.attributes import json_mode_for
from vlmfid.paths import ROOT, RUNS

# Una sola implementación del resumen y de la regla de la grilla: la de scripts/summarize_train.py
sys.path.insert(0, str(ROOT / "scripts"))
from summarize_train import read_jsonl, summarize_run, sweep_winners  # noqa: E402

pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

MODELS = {"llava_ov_05b": "LLaVA-OV-0.5B", "internvl3_1b": "InternVL3-1B",
          "qwen35_08b_base": "Qwen3.5-0.8B-Base", "vljepa_open_msrvtt": "VL-JEPA"}
VLMS = ["llava_ov_05b", "internvl3_1b", "qwen35_08b_base"]
VJ = "vljepa_open_msrvtt"
PROMPT = "p1_attributes"                      # el mismo en los 4 modelos (decisión del usuario, M §2.3)
ATTR_ES = {"shape": "forma", "x": "x", "y": "y", "color": "color", "pos": "posición (x e y)", "attr_mean": "attr_mean"}
RNG = np.random.default_rng(0)
print("runs:", RUNS)

In [ ]:
def load_run(run_dir):
    '''Una fila por imagen: verdad, atributos extraídos y aciertos (misma lógica que scripts/score.py y el notebook 02).'''
    run_dir = Path(run_dir)
    cfg = OmegaConf.load(run_dir / "config.yaml")
    if str(cfg.data.split) != "val":
        raise ValueError(f"{run_dir.name}: split {cfg.data.split}; este notebook es solo VAL")
    jm = json_mode_for(cfg)
    rows, seen = [], set()
    for r in read_jsonl(run_dir / "predictions.jsonl"):
        if r["id"] in seen:
            continue
        seen.add(r["id"])
        t, e = truth_from_record(r), extract(r["prediction"], jm)
        s = score(e, t)
        rows.append({"id": r["id"], "prediction": r["prediction"], "reference": r["reference"],
                     **{f"true_{a}": t[a] for a in ATTRIBUTES}, **{f"pred_{a}": e[a] for a in ATTRIBUTES},
                     **{f"ok_{m}": float(s[m]) for m in METRICS}, "n_words": e["n_words"]})
    df = pd.DataFrame(rows).set_index("id")
    df.attrs = {"model": str(cfg.model.name), "quant": str(cfg.model.quant), "prompt": str(cfg.prompt.id),
                "exp_id": str(cfg.exp_id), "stage": str(cfg.get("stage", "")), "json_mode": jm}
    return df


def score_val_predictions(path, json_mode, truth_by_id):
    '''Aciertos por imagen de un archivo val_predictions/step_XXXXX.jsonl del entrenamiento (no trae los latentes).'''
    rows = []
    for r in read_jsonl(path):
        s = score(extract(r["prediction"], json_mode), truth_by_id[r["id"]])
        rows.append({"id": r["id"], **{f"ok_{m}": float(s[m]) for m in METRICS}})
    return pd.DataFrame(rows).set_index("id")


def run_metrics(run_dir):
    p = Path(run_dir) / "metrics.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


def summary_row(df):
    out = {"model": df.attrs["model"], "quant": df.attrs["quant"], "n": len(df)}
    out |= {m: df[f"ok_{m}"].mean() for m in METRICS}
    out |= {f"omit_{a}": df[f"pred_{a}"].isna().mean() for a in ATTRIBUTES}
    out["words"] = df["n_words"].mean()
    return out


def paired_bootstrap(a, b, n_boot=2000):
    '''Diferencia media a − b sobre las mismas imágenes e IC 95 % por bootstrap pareado.'''
    d = (a - b).to_numpy()
    idx = RNG.integers(0, len(d), size=(n_boot, len(d)))
    boots = d[idx].mean(axis=1)
    return d.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def mcnemar(ok_a, ok_b):
    '''McNemar con corrección de continuidad sobre aciertos pareados (0/1). b = solo a acierta, c = solo b.'''
    b = int(((ok_a == 1) & (ok_b == 0)).sum())
    c = int(((ok_a == 0) & (ok_b == 1)).sum())
    if b + c == 0:
        return b, c, 0.0, 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, chi2, math.erfc(math.sqrt(chi2 / 2))


SHORT = re.compile(r"_m(\d+)$")


def train_runs(kind):
    '''kind: 'smoke' (_m6), 'grid' (_m150) o 'full' (sin _m<N>). Solo runs con config.yaml.'''
    out = []
    for d in sorted(RUNS.glob("train_*")):
        if not (d / "config.yaml").exists():
            continue
        m = SHORT.search(d.name)
        steps = int(m.group(1)) if m else None
        if (kind == "smoke" and steps == 6) or (kind == "grid" and steps == 150) or (kind == "full" and steps is None):
            out.append(d)
    return out


def summarize(dirs):
    rows = []
    for d in dirs:
        try:
            rows.append(summarize_run(d) | {"dir": d})
        except (FileNotFoundError, KeyError, ValueError) as e:
            print(f"se omite {d.name}: {e}")
    return pd.DataFrame(rows)

## 1. Prueba de humo (`configs/queues/stage2_smoke.txt`)

6 pasos con el lote efectivo real. Lo que se verifica (comentarios de la cola):

- **Entrenables** (`summary.json`): capas con LoRA por componente — VLM: 168 (LLaVA-OV, InternVL3) o 186 (Qwen3.5),
  `docs/resultados_etapa1.md` §7.2; VL-JEPA: 56 en el Predictor y 168 en el Y-Encoder.
- **Memoria pico**: decide **P8**. VL-JEPA necesita lote físico = efectivo (InfoNCE toma los negativos del lote); si el
  lote 128 no cabe en 24 GB, se baja a 64 **para los cuatro** (`configs/train/budget.yaml`).
- **La pérdida baja** entre el primer y el último paso, y el **paso 0 de val** (adaptador en cero) es coherente con el
  zero-shot de la etapa 1 (no es idéntico: el humo usa 32 imágenes de val en los VLM, no las 10 000).

In [ ]:
SMOKE = summarize(train_runs("smoke"))
EXPECTED_LORA = {"llava_ov_05b": {"lora_layers_llm": 168}, "internvl3_1b": {"lora_layers_llm": 168},
                 "qwen35_08b_base": {"lora_layers_llm": 186},
                 VJ: {"lora_layers_predictor": 56, "lora_layers_y_encoder": 168}}
if SMOKE.empty:
    print("la prueba de humo todavía no se ha corrido (configs/queues/stage2_smoke.txt)")
else:
    SMOKE["modelo"] = SMOKE["model"].map(MODELS)
    cols = ["status", "steps", "micro_batch", "peak_mem_gb", "loss_first", "loss_last", "sel_metric", "val_step0",
            "val_last"] + sorted(c for c in SMOKE.columns if c.startswith(("trainable", "lora_layers")))
    display(SMOKE.set_index(["modelo", "quant"])[cols])
    for _, r in SMOKE.iterrows():
        bad = {k: (r.get(k), v) for k, v in EXPECTED_LORA.get(r["model"], {}).items() if r.get(k) != v}
        down = r["loss_last"] is not None and r["loss_first"] is not None and r["loss_last"] < r["loss_first"]
        print(f"{r['modelo']:<20} {r['quant']:<5} capas LoRA {'OK' if not bad else 'DISTINTAS (obtenido, esperado): ' + str(bad)}"
              f" | pérdida {'baja' if down else 'NO baja'} | pico {r['peak_mem_gb']} GB")
    vj = SMOKE[SMOKE["model"] == VJ]
    if not vj.empty:
        print(f"\nP8 — VL-JEPA, lote físico {vj['micro_batch'].max()}: pico máximo {vj['peak_mem_gb'].max():.1f} GB de 24 GB "
              f"({'cabe' if vj['peak_mem_gb'].max() < 23 else 'NO cabe: bajar a 64 para los cuatro'})")

## 2. Grilla de hiperparámetros (`configs/train/grid.yaml`, G1)

LR {1e-4, 2e-4, 5e-4} × LoRA r {16, 64} (α = 16 fijo) × multiplicador del conector {1, 5}; 150 pasos × lote efectivo,
solo NF4, 500 imágenes de val (P1). **Regla:** por modelo gana la combinación con mejor métrica de selección en el
**último paso evaluado** (`attr_mean` máx. en los VLM; InfoNCE mín. en VL-JEPA), con `sweep_winners` de
`scripts/summarize_train.py`. El ganador va a `configs/train/models/<modelo>.yaml` (`train.lr`, `lora.r`,
`train.connector_lr_mult`) y el control bf16 lo hereda.

In [ ]:
GRID = summarize(train_runs("grid"))
WIN = {}
if GRID.empty:
    print("la grilla todavía no se ha corrido (configs/queues/stage2_grid*.txt)")
else:
    GRID["modelo"] = GRID["model"].map(MODELS)
    n_expected = 12
    for m, g in GRID.groupby("model"):
        incompletos = g[g["status"] != "completed"]["exp_id"].tolist()
        print(f"{MODELS[m]:<20} {len(g):>2}/{n_expected} combinaciones"
              + (f" | sin completar: {len(incompletos)}" if incompletos else ""))
    WIN = {m: r for (m, q), r in sweep_winners(GRID.to_dict("records")).items() if q == "nf4"}
    w = pd.DataFrame([{"modelo": MODELS[m], "train.lr": r["lr"], "lora.r": r["lora_r"],
                       "train.connector_lr_mult": r["connector_lr_mult"], r["sel_metric"]: r["val_last"],
                       "val_step0": r["val_step0"], "exp_id": r["exp_id"]} for m, r in WIN.items()]).set_index("modelo")
    print("\nGanador por modelo (-> configs/train/models/<modelo>.yaml):")
    display(w)

In [ ]:
# Mapa de la métrica final por modelo: filas LR, columnas r, un panel por multiplicador del conector
if not GRID.empty:
    mods = [m for m in [*VLMS, VJ] if m in set(GRID["model"])]
    mults = sorted(GRID["connector_lr_mult"].unique())
    fig, axes = plt.subplots(len(mods), len(mults), figsize=(4.2 * len(mults), 2.6 * len(mods)), squeeze=False)
    for i, m in enumerate(mods):
        g = GRID[GRID["model"] == m]
        minimize = g["sel_mode"].iloc[0] == "min"
        vmin, vmax = g["val_last"].min(), g["val_last"].max()
        for j, c in enumerate(mults):
            ax = axes[i, j]
            piv = g[g["connector_lr_mult"] == c].pivot(index="lr", columns="lora_r", values="val_last")
            im = ax.imshow(piv.to_numpy(), cmap="viridis_r" if minimize else "viridis", vmin=vmin, vmax=vmax, aspect="auto")
            ax.set_xticks(range(piv.shape[1]), [f"r={x}" for x in piv.columns])
            ax.set_yticks(range(piv.shape[0]), [f"{x:g}" for x in piv.index])
            for a in range(piv.shape[0]):
                for b in range(piv.shape[1]):
                    v = piv.iat[a, b]
                    if pd.notna(v):
                        ax.text(b, a, f"{v:.3f}", ha="center", va="center", fontsize=8, color="w")
            ax.set_title(f"{MODELS[m]} — conector ×{c:g} ({g['sel_metric'].iloc[0]}, {'mín' if minimize else 'máx'})", fontsize=8)
            ax.grid(False)
    plt.tight_layout()
    plt.show()

    # Sensibilidad: cuánto cambia la métrica al mover un solo eje (promedio sobre los otros dos)
    sens = []
    for m, g in GRID.groupby("model"):
        row = {"modelo": MODELS[m], "rango_total": g["val_last"].max() - g["val_last"].min()}
        for ax_ in ("lr", "lora_r", "connector_lr_mult"):
            means = g.groupby(ax_)["val_last"].mean()
            row[f"efecto_{ax_}"] = means.max() - means.min()
        sens.append(row)
    print("Efecto de cada eje (diferencia entre la mejor y la peor media marginal):")
    display(pd.DataFrame(sens).set_index("modelo"))

In [ ]:
# Curvas de val de cada combinación (paso 0, 50, 100, 150): ¿el ganador ya estaba arriba o cruza al final?
if not GRID.empty:
    mods = [m for m in [*VLMS, VJ] if m in set(GRID["model"])]
    fig, axes = plt.subplots(1, len(mods), figsize=(4.6 * len(mods), 3.4), squeeze=False)
    for ax, m in zip(axes[0], mods):
        g = GRID[GRID["model"] == m]
        metric = g["sel_metric"].iloc[0]
        for _, r in g.iterrows():
            v = pd.DataFrame(read_jsonl(r["dir"] / "val_log.jsonl"))
            if v.empty:
                continue
            is_w = m in WIN and WIN[m]["exp_id"] == r["exp_id"]
            ax.plot(v["step"], v[metric], marker=".", lw=2.2 if is_w else 0.8, alpha=1 if is_w else 0.6,
                    label=f"lr {r['lr']:g} r{r['lora_r']} c×{r['connector_lr_mult']:g}" + (" ★" if is_w else ""))
        ax.set(title=f"{MODELS[m]}", xlabel="paso", ylabel=metric)
        ax.legend(fontsize=5.5, ncol=2)
    plt.tight_layout()
    plt.show()

## 3. ¿Es robusto el ganador de la grilla? (VLM)

La grilla compara 12 combinaciones sobre las **mismas 500 imágenes** de val. Con `val_predictions/step_00150.jsonl`
se re-puntúa cada imagen y se estima el margen del ganador sobre el segundo con un bootstrap pareado. Si el IC
incluye el 0, el ganador no se distingue del segundo con 500 imágenes: se declara, pero la regla se mantiene (no
se elige a mano). VL-JEPA no guarda predicciones en val (su métrica es la InfoNCE del lote); se reporta solo su
margen.

In [ ]:
VAL_TRUTH = {r["id"]: truth_from_record(r) for r in load_manifest("val")}
if not GRID.empty:
    rows = []
    for m, g in GRID.groupby("model"):
        g = g[g["val_last"].notna()].sort_values("val_last", ascending=(g["sel_mode"].iloc[0] == "min"))
        if len(g) < 2:
            continue
        first, second = g.iloc[0], g.iloc[1]
        row = {"modelo": MODELS[m], "ganador": f"lr {first['lr']:g} r{first['lora_r']} c×{first['connector_lr_mult']:g}",
               "segundo": f"lr {second['lr']:g} r{second['lora_r']} c×{second['connector_lr_mult']:g}",
               "métrica": first["sel_metric"], "ganador_v": first["val_last"], "segundo_v": second["val_last"],
               "margen": first["val_last"] - second["val_last"]}
        if m != VJ:
            step = int(first["val_last_step"])
            pa, pb = (d["dir"] / "val_predictions" / f"step_{step:05d}.jsonl" for d in (first, second))
            if pa.exists() and pb.exists():
                jm = json_mode_for(OmegaConf.load(first["dir"] / "config.yaml"))
                a = score_val_predictions(pa, jm, VAL_TRUTH)["ok_attr_mean"]
                b = score_val_predictions(pb, jm, VAL_TRUTH)["ok_attr_mean"].reindex(a.index)
                d, lo, hi = paired_bootstrap(a, b)
                row |= {"n": len(a), "IC95_lo": lo, "IC95_hi": hi, "robusto": lo > 0}
        rows.append(row)
    display(pd.DataFrame(rows).set_index("modelo"))

## 4. Ajuste completo (`configs/queues/stage2_train.txt`)

1 época de train (125 000 imágenes, 977 pasos con lote 128), checkpoint elegido en val (500 imágenes). Arriba, la
pérdida de entrenamiento (entropía cruzada por token en los VLM, InfoNCE bidireccional en VL-JEPA) y la norma del
gradiente; abajo, la métrica de selección en val con el paso elegido (`adapter_best`).

In [ ]:
FULL = summarize(train_runs("full"))
if FULL.empty:
    print("el ajuste completo todavía no se ha corrido (configs/queues/stage2_train.txt)")
else:
    FULL["modelo"] = FULL["model"].map(MODELS)
    display(FULL.set_index(["modelo", "quant"])[["status", "lr", "lora_r", "connector_lr_mult", "steps", "peak_mem_gb",
                                                 "loss_first", "loss_last", "sel_metric", "val_step0", "best_step",
                                                 "best_value", "val_last"]])
    hp = json.loads((FULL["dir"].iloc[0] / "summary.json").read_text(encoding="utf-8")) \
        if (FULL["dir"].iloc[0] / "summary.json").exists() else {}
    if hp and not hp.get("hparams_confirmed", False):
        print("AVISO: hparams_confirmed = false en summary.json (no debería pasar en un ajuste completo)")

In [ ]:
if not FULL.empty:
    mods = [m for m in [*VLMS, VJ] if m in set(FULL["model"])]
    fig, axes = plt.subplots(3, len(mods), figsize=(4.6 * len(mods), 9), squeeze=False)
    for j, m in enumerate(mods):
        for _, r in FULL[FULL["model"] == m].iterrows():
            tr = pd.DataFrame(read_jsonl(r["dir"] / "train_log.jsonl"))
            v = pd.DataFrame(read_jsonl(r["dir"] / "val_log.jsonl"))
            ls = "-" if r["quant"] == "nf4" else "--"
            if not tr.empty:
                axes[0, j].plot(tr["step"], tr["loss"], ls, lw=0.9, label=r["quant"])
                axes[1, j].plot(tr["step"], tr["grad_norm"], ls, lw=0.9, label=r["quant"])
            if not v.empty:
                metric = r["sel_metric"]
                line, = axes[2, j].plot(v["step"], v[metric], ls, marker=".", label=r["quant"])
                if r["best_step"] is not None:
                    axes[2, j].axvline(r["best_step"], color=line.get_color(), ls=":", lw=1)
                axes[2, j].set_ylabel(metric)
        axes[0, j].set_title(MODELS[m], fontsize=9)
        axes[0, j].set_ylabel("pérdida (train)")
        axes[1, j].set_ylabel("‖g‖ (antes del recorte)")
        axes[2, j].set_xlabel("paso")
        for i in range(3):
            axes[i, j].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
# VLM: atributos en val a lo largo del ajuste (500 imágenes) y fracción de respuestas cortadas por longitud;
# VL-JEPA: top-1 dentro del lote en ambos sentidos (azar = 1/128).
if not FULL.empty:
    for _, r in FULL[FULL["quant"] == "nf4"].iterrows():
        v = pd.DataFrame(read_jsonl(r["dir"] / "val_log.jsonl"))
        if v.empty:
            continue
        fig, ax = plt.subplots(figsize=(6, 3))
        cols = ["acc_p2t", "acc_t2p"] if r["model"] == VJ else [*ATTRIBUTES, "attr_mean", "truncated"]
        for c in cols:
            if c in v:
                ax.plot(v["step"], v[c], marker=".", label=ATTR_ES.get(c, c))
        ax.set(title=f"{MODELS[r['model']]} (NF4) — val durante el ajuste", xlabel="paso")
        ax.legend(fontsize=7, ncol=3)
        plt.show()

## 5. Val completo: zero-shot frente a ajustado (`configs/queues/stage2_val.txt`)

El ajuste elige el paso con 500 imágenes; el número que se reporta en val es el de las **10 000** (P1), con el mismo
`infer.py` y el mismo extractor que la etapa 1. La referencia zero-shot es **`p1_attributes`** (el prompt de la etapa
2), no el mejor prompt zero-shot de cada modelo: así la diferencia mide el efecto del ajuste y no el del prompt.
Para VL-JEPA, la referencia zero-shot usa el decodificador genérico (MSR-VTT) y el ajustado su Y-Decoder de la etapa 2:
la diferencia mezcla el ajuste del modelo con el del lector; la condición intermedia, que separaría ambas cosas, se canceló en el cierre de la etapa 1 (CE1).

In [ ]:
def load_if(path):
    return load_run(path) if (Path(path) / "predictions.jsonl").exists() else None


ZS, FT = {}, {}
for m in [*VLMS, VJ]:
    for q in ("nf4", "bf16"):
        z = load_if(RUNS / f"zeroshot_{m}_{q}_{PROMPT}_val")
        f = load_if(RUNS / f"finetuned_{m}_{q}_{PROMPT}_val")
        if z is not None:
            ZS[(m, q)] = z
        if f is not None:
            FT[(m, q)] = f
INTER = load_if(RUNS / f"zeroshot_{VJ}_nf4_{PROMPT}_val_decm3di")
print("zero-shot p1:", sorted(ZS), "\najustados:", sorted(FT), "\nintermedia de VL-JEPA:", INTER is not None)
bad = [k for k, df in {**{('zs',) + k: v for k, v in ZS.items()}, **{('ft',) + k: v for k, v in FT.items()}}.items()
       if len(df) != 10_000]
print("runs con n != 10 000 (no comparables):", bad or "ninguno")

In [ ]:
train_recs, val_recs = load_manifest("train"), load_manifest("val")
BASE_MAJ = majority_baseline(train_recs, val_recs)
rows = []
for (m, q), f in FT.items():
    if q != "nf4" or (m, q) not in ZS:
        continue
    z = ZS[(m, q)]
    common = f.index.intersection(z.index)
    for met in METRICS:
        a, b = f.loc[common, f"ok_{met}"], z.loc[common, f"ok_{met}"]
        d, lo, hi = paired_bootstrap(a, b)
        bb, cc, _, pv = mcnemar(a, b) if met != "attr_mean" else (None, None, None, None)
        rows.append({"modelo": MODELS[m], "métrica": ATTR_ES[met], "zero-shot": b.mean(), "ajustado": a.mean(),
                     "Δ": d, "IC95": f"[{lo:+.3f}, {hi:+.3f}]", "mayoritaria": BASE_MAJ.get(met),
                     "b (solo ajust.)": bb, "c (solo zs)": cc, "p McNemar": pv})
if rows:
    DELTA = pd.DataFrame(rows)
    display(DELTA.set_index(["modelo", "métrica"]))
    piv = DELTA.pivot(index="modelo", columns="métrica", values="ajustado")
    piv0 = DELTA.pivot(index="modelo", columns="métrica", values="zero-shot")
    order = [ATTR_ES[a] for a in [*ATTRIBUTES, "attr_mean"]]
    fig, ax = plt.subplots(figsize=(10, 3.6))
    x = np.arange(len(order))
    w = 0.8 / max(len(piv), 1)
    for i, mod in enumerate(piv.index):
        ax.bar(x + i * w, piv.loc[mod, order], w, label=f"{mod} ajustado")
        ax.scatter(x + i * w, piv0.loc[mod, order], marker="_", s=200, color="k", zorder=3,
                   label="zero-shot (p1)" if i == 0 else None)
    ax.scatter(x + 0.4 - w / 2, [BASE_MAJ.get(k) for k in [*ATTRIBUTES, "attr_mean"]], marker="x", color="r",
               label="clase mayoritaria", zorder=3)
    ax.set_xticks(x + 0.4 - w / 2, order)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, ncol=3)
    ax.set_title("Exactitud por atributo en val (NF4): ajustado (barras) y zero-shot con p1_attributes (–)", fontsize=9)
    plt.show()
else:
    print("faltan los runs de val completo de la etapa 2 (configs/queues/stage2_val.txt) o los zero-shot p1 de la etapa 1")

In [ ]:
# VL-JEPA: zero-shot (lector genérico) -> intermedia (lector ajustado a M3DI) -> ajustado (modelo y lector de la etapa 2)
steps = [("zero-shot, decodificador genérico", ZS.get((VJ, "nf4"))), ("intermedia, decodificador M3DI", INTER),
         ("ajustado, decodificador de la etapa 2", FT.get((VJ, "nf4")))]
steps = [(k, v) for k, v in steps if v is not None]
if steps:
    display(pd.DataFrame([{"condición": k, **{ATTR_ES[m]: v[f"ok_{m}"].mean() for m in METRICS}} for k, v in steps])
            .set_index("condición"))
    if FT.get((VJ, "nf4")) is not None:
        for _, r in FT[(VJ, "nf4")].sample(5, random_state=0).iterrows():
            print(f"  ref: {r['reference']}\n  dec: {r['prediction']}\n")
else:
    print("sin runs de VL-JEPA en val todavía")

## 6. Omitir frente a equivocarse, antes y después del ajuste

El ajuste enseña el **formato** del caption de M3DI. Si la ganancia viene sobre todo de bajar las omisiones, es
aprendizaje de formato; si sube la exactitud **condicional** (entre las respuestas que sí nombran el atributo), hay
además mejora de percepción o de vocabulario.

In [ ]:
rows = []
for (m, q), f in FT.items():
    if q != "nf4":
        continue
    for tag, df in (("zero-shot", ZS.get((m, q))), ("ajustado", f)):
        if df is None:
            continue
        for a in ATTRIBUTES:
            said = df[f"pred_{a}"].notna()
            rows.append({"modelo": MODELS[m], "condición": tag, "atributo": ATTR_ES[a], "exactitud": df[f"ok_{a}"].mean(),
                         "omitida": 1 - said.mean(), "condicional": df.loc[said, f"ok_{a}"].mean() if said.any() else np.nan})
if rows:
    OM = pd.DataFrame(rows).set_index(["modelo", "atributo", "condición"]).unstack("condición")
    display(OM)

In [ ]:
# Confusiones del ajustado (NF4), forma, x e y; normalizadas por fila (∅ = omitido)
for a in ("shape", "x", "y"):
    keys = [k for k in FT if k[1] == "nf4"]
    if not keys:
        break
    fig, axes = plt.subplots(1, len(keys), figsize=(4.8 * len(keys), 3.8), squeeze=False)
    for ax, (m, q) in zip(axes[0], keys):
        df = FT[(m, q)]
        ct = pd.crosstab(df[f"true_{a}"], df[f"pred_{a}"].fillna("∅"), normalize="index")
        cols = [c for c in ct.index if c in ct.columns] + [c for c in ct.columns if c not in ct.index]
        ct = ct.reindex(columns=cols, fill_value=0)
        ax.imshow(ct.to_numpy(), vmin=0, vmax=1, cmap="Blues", aspect="auto")
        ax.set_xticks(range(ct.shape[1]), ct.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(ct.shape[0]), ct.index, fontsize=7)
        for i in range(ct.shape[0]):
            for j in range(ct.shape[1]):
                if ct.iat[i, j] >= 0.05:
                    ax.text(j, i, f"{ct.iat[i, j]:.2f}", ha="center", va="center", fontsize=6,
                            color="w" if ct.iat[i, j] > 0.5 else "k")
        ax.set_title(f"{ATTR_ES[a]} — {MODELS[m]} ajustado", fontsize=9)
        ax.grid(False)
    plt.tight_layout()
    plt.show()

## 7. Control bf16 después del ajuste

Misma corrida LoRA sobre base bf16 (D6), con la combinación ganadora de NF4. Cada adaptador se evalúa con la
precisión con que se entrenó. La pregunta: ¿la brecha NF4 − bf16 de la etapa 1 se cierra, se mantiene o crece tras
el ajuste? (QLoRA [8] reporta que el ajuste en NF4 recupera el rendimiento de 16 bits, en modelos de 7B–65B.)

In [ ]:
rows = []
for m in [*VLMS, VJ]:
    for tag, src in (("zero-shot", ZS), ("ajustado", FT)):
        a, b = src.get((m, "nf4")), src.get((m, "bf16"))
        if a is None or b is None:
            continue
        common = a.index.intersection(b.index)
        d, lo, hi = paired_bootstrap(a.loc[common, "ok_attr_mean"], b.loc[common, "ok_attr_mean"])
        row = {"modelo": MODELS[m], "condición": tag, "nf4": a.loc[common, "ok_attr_mean"].mean(),
               "bf16": b.loc[common, "ok_attr_mean"].mean(), "nf4−bf16": d, "IC95": f"[{lo:+.3f}, {hi:+.3f}]"}
        for x in ATTRIBUTES:
            bb, cc, _, pv = mcnemar(a.loc[common, f"ok_{x}"], b.loc[common, f"ok_{x}"])
            row[f"p_{x}"] = pv
        rows.append(row)
if rows:
    display(pd.DataFrame(rows).set_index(["modelo", "condición"]))
else:
    print("faltan pares NF4/bf16 en val")

## 8. VL-JEPA: Y-Decoder de la etapa 2 y pérdida de información

### 8a. Y-Decoder sobre el Y-Encoder ajustado (`configs/queues/ydecoder_msrvtt.txt` §4)

Continúa desde el genérico (`_cont`) con captions de M3DI; el paso se elige por el techo en val (decodificar el `S_Y`
del caption verdadero). Se compara con el decodificador de la condición intermedia (sobre el Y-Encoder sin ajustar).

In [ ]:
YD = {}
for d in sorted(RUNS.glob(f"ydec_{VJ}_*_k*_r*_e*_cont")):
    log = d / "val_log.jsonl"
    if log.exists() and "humo" not in d.name:
        YD[d.name] = pd.DataFrame(read_jsonl(log))
if YD:
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for name, df in YD.items():
        ax.plot(df["step"], df["attr_mean"], marker=".", label=name.replace(f"ydec_{VJ}_", ""))
    ax.set(title="techo en val (attr_mean del S_Y verdadero decodificado)", xlabel="paso")
    ax.legend(fontsize=7)
    plt.show()
    display(pd.DataFrame([{"run": n, "etapa": "2 (Y-Encoder ajustado)" if "_ft_" in n else "intermedia",
                           "mejor_paso": int(df.loc[df["attr_mean"].idxmax(), "step"]),
                           **{ATTR_ES[a]: df.loc[df["attr_mean"].idxmax(), a] for a in [*ATTRIBUTES, "attr_mean"] if a in df}}
                          for n, df in YD.items()]).set_index("run"))
else:
    print("sin decodificadores ajustados a M3DI todavía")

### 8b. Pérdida de información por etapa: etapa 1 frente a etapa 2 (`docs/metodologia.md` §2.5)

Mismas sondas, imágenes y semillas en ambas etapas. Puntos: E1 (X-Encoder), E2 (Predictor), E3 (Ŝ_Y), T1 (S_Y del
caption), E4/T2 (texto decodificado). Información en bits (recortada a [0, H]); las pérdidas son diferencias entre
puntos. Δ = etapa 2 − etapa 1: si el ajuste ayuda, la información en E3 y E4 sube y las pérdidas bajan. E1 no debería
cambiar (el X-Encoder está congelado): es un control.

**Nota (CE1):** la pérdida de información de la etapa 1 se canceló en el cierre. Para tener la comparación hay
que correr antes `configs/queues/stage1_infoloss.txt` (solo inferencia); si no, esta sección muestra solo la etapa 2.


In [ ]:
def load_info(tag):
    p = RUNS / (f"infoloss_{VJ}_nf4_{PROMPT}" + (f"_{tag}" if tag else "")) / "tabla_discretos.csv"
    return pd.read_csv(p).set_index("factor") if p.exists() else None


I1, I2 = load_info(None), load_info("ft")
for name, t in (("etapa 1", I1), ("etapa 2", I2)):
    print(name, "—", "sin correr" if t is None else "")
    if t is not None:
        display(t)
if I1 is not None and I2 is not None:
    common = [c for c in I1.columns if c in I2.columns and c != "H_bits"]
    D = (I2[common] - I1[common])
    print("Δ = etapa 2 − etapa 1")
    display(D.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-1, vmax=1))
    info_cols = [c for c in common if c.startswith("info_")]
    fig, axes = plt.subplots(1, len(I1.index), figsize=(3.2 * len(I1.index), 3), sharey=True)
    for ax, f in zip(np.atleast_1d(axes), I1.index):
        ax.plot([c[5:] for c in info_cols], I1.loc[f, info_cols], marker="o", label="etapa 1")
        ax.plot([c[5:] for c in info_cols], I2.loc[f, info_cols], marker="s", label="etapa 2")
        ax.axhline(I1.loc[f, "H_bits"], color="k", ls=":", lw=0.8)
        ax.set_title(f"{f} (H = {I1.loc[f, 'H_bits']:.2f} bits)", fontsize=8)
    np.atleast_1d(axes)[0].set_ylabel("información (bits)")
    np.atleast_1d(axes)[0].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## 9. Tabla resumen

### 9a. Val (10 000 imágenes), prompt `p1_attributes`

NF4 es la condición principal; bf16, el control. Comparación válida: **entre nuestros 4 modelos bajo el mismo
protocolo**, no contra cifras de los papers (modelos de 7B o más).

In [ ]:
R = []
for m in [*VLMS, VJ]:
    for q in ("nf4", "bf16"):
        for tag, src, prefix in (("zero-shot", ZS, "zeroshot"), ("ajustado", FT, "finetuned")):
            df = src.get((m, q))
            if df is None:
                continue
            met = run_metrics(RUNS / f"{prefix}_{m}_{q}_{PROMPT}_val") or {}
            R.append({"modelo": MODELS[m], "precisión": q, "condición": tag,
                      **{ATTR_ES[x]: df[f"ok_{x}"].mean() for x in METRICS}, **{k: met.get(k) for k in TEXT_METRICS}})
R.append({"modelo": "clase mayoritaria", "precisión": "—", "condición": "—",
          **{ATTR_ES[a]: BASE_MAJ[a] for a in ATTRIBUTES}, "attr_mean": BASE_MAJ["attr_mean"]})
RES = pd.DataFrame(R).set_index(["modelo", "precisión", "condición"])
display(RES)
out = RUNS / "_tablas" / "etapa2_val_resumen.csv"
if len(R) > 1:
    out.parent.mkdir(parents=True, exist_ok=True)
    RES.to_csv(out)
    print("tabla en", out)

### 9b. Test final (solo lectura)

`scripts/final_test.py` corre el plan congelado de `configs/final_test.yaml` **una sola vez** y deja
`runs/final_test/tabla_final.csv`. Esta celda solo lo muestra; nada de lo de arriba debe cambiarse después de verlo.

In [ ]:
p = RUNS / "final_test" / "tabla_final.csv"
if p.exists():
    display(pd.read_csv(p))
    ledger = RUNS / "final_test" / "ledger.jsonl"
    if ledger.exists():
        print("lanzamientos registrados:", len(read_jsonl(ledger)))
else:
    print("test todavía no se ha corrido (correcto hasta cerrar todo lo de val)")